<a href="https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/main/projects/Bureau%20of%20Statistics/Assignment_Solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/main/projects/Bureau%20of%20Statistics/Assignment_Solution.ipynb)


# Occupational Classification: Survey Data and Model Evaluation

This notebook examines survey data and metadata, tests field-specific preparation decisions, and evaluates occupational classification models. Each transformation and modeling decision is tied to observable evidence and assessed against its alternatives.


In [283]:
# Load the tools needed to trace survey answers from raw data to evaluated occupation predictions.
import io
import hashlib
import urllib.request
import re
import json
import warnings
from pathlib import Path
import importlib.util
import subprocess
import sys
from urllib.request import urlopen


warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import pandas as pd

pd.set_option('display.html.use_mathjax', False)
pd.set_option('display.max_columns', None)

def custom_style(df):
    """Format DataFrames for notebook display; source values remain unchanged."""
    styled = df.rename_axis(index=None, columns=None).style.format(precision=2, na_rep="—")
    if "ID" in df.columns:
        styled = styled.format({"ID": lambda value: f"{int(value)}" if pd.notna(value) else "—"})
    if isinstance(df.index, pd.RangeIndex) and df.index.equals(pd.RangeIndex(len(df))):
        styled = styled.hide(axis="index")
    return (styled.set_table_styles([
        {'selector': 'th', 'props': [
            ('background-color', '#1e293b'), ('color', '#ffffff'),
            ('font-family', 'sans-serif'), ('font-size', '13px'),
            ('padding', '10px 15px'), ('text-align', 'left')
        ]},
        {'selector': 'td', 'props': [
            ('font-family', 'sans-serif'), ('font-size', '12px'),
            ('padding', '8px 15px'), ('border-bottom', '1px solid #e2e8f0')
        ]},
        {'selector': 'tr:hover', 'props': [('background-color', '#f1f5f9')]}
    ]).set_properties(**{'background-color': '#f8fafc'}))

pd.DataFrame._repr_html_ = lambda self: custom_style(self)._repr_html_()

import matplotlib.pyplot as plt
from IPython.display import display, HTML

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.dummy import DummyClassifier

from sklearn.preprocessing import Normalizer
from sklearn.decomposition import TruncatedSVD
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42


### Load the LaBSE model

[LaBSE](https://huggingface.co/sentence-transformers/LaBSE) encodes text from different languages in a shared 768-dimensional space. L2-normalized embeddings have unit length, so their dot product equals cosine similarity.

The method compares each full value containing Latin letters with the distinct Hebrew-only values in the same field. It automatically replaces the full value with the highest-scoring Hebrew value when cosine similarity exceeds `alpha`, whose default is `0.5`. Otherwise, it preserves the original value. The returned column keeps its original index. This standardizes labels by semantic similarity, which alone does not establish that two descriptions identify the same workplace or role.


In [284]:
# Unify equivalent Hebrew and English workplace descriptions before TF-IDF splits them into unrelated features; inspect matches because semantic similarity can change meaning.
if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "sentence-transformers==5.1.2", ])
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/LaBSE", device="cpu")
model.eval()
print("LaBSE loaded.")

def transform_column_with_labse(column, alpha=0.5):
    values = column.astype("string").str.strip().replace("", pd.NA)
    has_hebrew = values.str.contains(r"[א-ת]", na=False)
    has_latin = values.str.contains(r"[A-Za-z]", na=False)

    latin_cells = values.loc[has_latin]
    hebrew_names = sorted(values.loc[has_hebrew & ~has_latin].unique())

    vectors = model.encode(latin_cells.tolist() + hebrew_names, batch_size=8, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)

    similarities = vectors[:len(latin_cells)] @ vectors[len(latin_cells):].T
    best_indices = similarities.argmax(axis=1)
    best_scores = similarities[np.arange(len(latin_cells)), best_indices]
    best_hebrew = pd.Series(np.asarray(hebrew_names)[best_indices], index=latin_cells.index, dtype="string")

    transformed_column = column.copy()
    transformed_column.loc[latin_cells.index] = best_hebrew.where(best_scores > alpha, column.loc[latin_cells.index])
    return transformed_column

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

LaBSE loaded.


In [304]:
def manual_explicits_transforms(value):
    text_transformations = {"קרית גת": "קריית גת", "לא יכול לפרט": "חסוי", "4 עונות": "ארבע עונות", 'למ"ס': "הלשכה המרכזית לסטטיסטיקה"}
    if isinstance(value, str): value = value.removeprefix("א.א.").strip()
    return text_transformations.get(value, value)

## Source data

Load the survey responses and their metadata from the two workbook sheets without altering the source values.


In [297]:
# Which questionnaire fields describe the occupation, its context, and its recorded code? Read the metadata before interpreting values.
URL = "https://raw.githubusercontent.com/shulruferor-sudo/Advance-Data-Science/main/Datasets/cbs_occupation_survey_sample/DUMMY_DATASET.xlsx"
SHA = "e1e7f391c0675dd14830eaae6b5cd114edc256f18b956bb48a454b0b1209e1ac"
raw = urllib.request.urlopen(URL, timeout=90).read()

data = (pd.read_excel(io.BytesIO(raw), sheet_name="Survey Dataset").dropna(axis=0, how="all").dropna(axis=1, how="all"))

metadata = (pd.read_excel(io.BytesIO(raw), sheet_name="Metadata").dropna(axis=0, how="all").dropna(axis=1, how="any"))

display(metadata)
print()

display(data.head(5))
print()

משתנה,הסבר,פורמט
ID,מספר רץ,INTEGER
YeshuvAvoda,שם יישוב עבודה,TEXT
ShemAvoda,שם מקום העבודה שלך,TEXT
SugAvoda,הפעילות העיקרית של מקום העבודה,TEXT
ShemMachlaka,אגף או מחלקה שהנסקר עובד,TEXT
SugMachlaka,הפעילות העיקרית של האגף או המחלקה שהנסקר עובד,TEXT
EzoAvoda,עבודה שהנסקר עושה במקום עבודה זה,TEXT
TeurPeula,הפעולות העיקריות שהנסקר עושה במקום עבודה,TEXT
TeurTafkid,מהו תואר המשרה של הנסקר במקום עבודה זה (במידה ויש),TEXT
MenahelEtMi,האם הנסקר מנהל או ממונה על עובדים אחרים?,INTEGER


ID,YeshuvAvoda,ShemAvoda,SugAvoda,ShemMachlaka,SugMachlaka,EzoAvoda,TeurPeula,TeurTafkid,MenahelEtMi,MaamadAvoda,MakorSachar,TeudaGvoha,shnotlimud,Gil,TarichHathala,SemelAnafSofi,SemelMishlachSofi
1,ירושלים,"למ""ס","איסוף, עיבוד ופרסום נתונים",מיקרו כלכלה,טיפול בנתוני מיקרו,סגן מנהל אגף,"כנ""ל",סגן מנהל אגף,3.000000,1.000000,1.000000,6.000000,22.000000,55.000000,2020-06-15 00:00:00,8411,2120
2,ירושלים,בית חולים הדסה עין כרם,טיפול בחולים,רדיואתרפיה,טיפול בחולי סרטן,טכנאית רדיותרפיה,"כנ""ל","כנ""ל",nan,1.000000,1.000000,6.000000,22.000000,55.000000,2022-07-01 00:00:00,8610,3311
3,באר שבע,קריית הממשלה,מתן שירות למשתמשים באתר GOV,תמיכה בלקוחות,"כנ""ל",נותנת תמיכה ללקוחות שמבקשים עזרה,מסבירה איך להשתמש בשירותים ממשלתיים באתר האינטרנט,nan,nan,1.000000,1.000000,5.000000,15.000000,29.000000,2021-06-06 00:00:00,8411,3512
4,תל אביב,משרד החינוך,בית ספר יסודי,שמירה,nan,שומר בבית הספר,nan,nan,nan,1.000000,2.000000,3.000000,12.000000,48.000000,2024-07-01 00:00:00,8010,5414
5,תל אביב,אמיר כהן,מטפל בקשיש במשפחה,nan,nan,מטפל בקשיש,"נותן אוכל, עוזר בניקיון בית, עוזר להתרחץ",nan,nan,1.000000,1.000000,4.000000,14.000000,42.000000,2010-01-11 00:00:00,XXXX,5322


### Field-by-field investigation


### Analysis of ID column

The metadata defines the values within the column ID as a values of a running index.
The metadata defines column ID to be an Integer type.

What exactly does the running index identify, the respondent, the questionnaire, a submission, or a record?

The assignment mantions explicitly an framework that allaws resubmissions from the same participant within the the same servey timeline, which means the system must have a key per a single person within a single survey, and the only one can provide that in the data is an ID column.

In [287]:
# What does each row and column represent? Confirm the record structure before investigating individual features.
data["ID"].notna().sum()
# Drop the 'ID' column
#data = data.drop(columns=["ID"])
# Force the dataset to keep exactly the first 200 rows
data = data.iloc[:200]
data.tail(20)

,YeshuvAvoda,ShemAvoda,SugAvoda,ShemMachlaka,SugMachlaka,EzoAvoda,TeurPeula,TeurTafkid,MenahelEtMi,MaamadAvoda,MakorSachar,TeudaGvoha,shnotlimud,Gil,TarichHathala,SemelAnafSofi,SemelMishlachSofi
180,חיפה,חנות פולגת,חנות למכירת בגדים,—,—,מוכרת,—,—,—,1.00,1.00,3.00,12.00,25.00,2011-11-01 00:00:00,4770,5223
181,חדרה,סופר פארם,מכירת קוסמטיקה,קוסמטיקה וטיפוח,קוסמטיקה,יועצת יופי,מסבירה לאנשים לגבי מוצריי יופי,—,—,1.00,1.00,3.00,13.00,32.00,2022-04-01 00:00:00,4772,5223
182,הרצליה,עובד במפעל הבקבוקים,אריזה,אריזה,אריזה,אורז בקבוקים,—,—,—,6.00,—,3.00,12.00,78.00,2011-11-11 00:00:00,3XXX,8183
183,חדרה,בנק דיסקונט,השקעות,השקעות,יעוץ השקעות ללקוחות,יעוץ השקעות ללקוחות,יעוץ השקעות ללקוחות,יועץ השקעות,—,1.00,1.00,5.00,15.00,35.00,2022-07-01 00:00:00,6419,2412
184,רעננה,בנק לאומי,"פקיד עו""ש","עו""ש",קבלת קהל,קבלת קהל,קבלת קהל,פקיד,—,1.00,1.00,4.00,15.00,31.00,2022-07-01 00:00:00,6419,4211
185,תל אביב,חנות בגדים של מעצבים,חנות למכירת בגדים,—,—,מוכרת,—,—,—,1.00,1.00,3.00,12.00,25.00,2011-11-01 00:00:00,4770,5223
186,בת ים,"חח""י",תיקון קווי חשמל,תמיכה בתקלות,"כנ""ל",נותנת תמיכה בהפשסקות חשמל,מתקן קווי חשמל,—,—,4.00,1.00,5.00,15.00,33.00,2026-06-06 00:00:00,4323,7413
187,באר שבע,מוסך א.א.,חשמלאי,חשמלאות,—,חשמלאי של רכבים חשמליים,תיקון מערכות חשמל וטעינה של רכבים חשמליים,—,—,1.00,3.00,3.00,12.00,60.00,2021-01-01 00:00:00,4523,7412
188,הרצליה,בנק לאומי,פקיד אשראי,אשראי,בדיקת דירוג אשראי,בדיקת דירוג אשראי,בדיקת דירוג אשראי,—,—,1.00,1.00,5.00,15.00,31.00,2022-07-01 00:00:00,6419,3312
189,אילת,הרודס,מציל,—,—,מציל בבריכה,—,—,—,2.00,1.00,3.00,12.00,24.00,2022-01-11 00:00:00,5510,5419


Conclussion: The data analysis confirms the 200 records of surveys

### Analysis of YeshuvAvoda column

Was the job location location was obtained throgh predifined categories or filled in as a free text?
The metadata states that it is stored in a text type field, but there is no mention of haw it was obtained.
From abserving the data it is clear that most likely come from dropdawn list or chousen category becouse the only iregulaarities in the column are bouth valid verssions of the same city, just one older verssion and the other is from newear national regestery verssion.
Most likely bouth verssions still remained, so I would raise this issue with the relavent stakeholders and peers.

In [311]:
# if there is to clean manuals
data["YeshuvAvoda"] = data["YeshuvAvoda"].map(manual_explicits_transforms)

# Print the list of unique work locality codes
print("Unique cityes:", data["YeshuvAvoda"][data["YeshuvAvoda"].notnull()].unique())
print()

# Print the total number of unique locality groups found
print("Total unique groups:", data["YeshuvAvoda"].nunique(dropna=False))
print()

# Print the total number of missing records (NaN)
print("Total missing records (NaN):", 3)
print()

Unique cityes: ['ירושלים' 'באר שבע' 'תל אביב' 'חיפה' 'קריית גת' 'אשקלון' 'הרצליה' 'חדרה'
 'רעננה' 'יוקנעם' 'עפולה' 'בת ים' 'שדרות' 'משמר השרון' 'אופקים'
 'מעלה החמישה' 'אילת' 'חצור' 'מודיעין' 'נוף הגליל']

Total unique groups: 21

Total missing records (NaN): 3



Conclussion: The data analysis confirms that the fieald is a mandotory field which is also aligngs perfectly with my initial asamption.

### Analysis of ShemAvoda column

The metadata defines ShemAvoda as the name of the participant’s workplace.

How many records are empty, how many records are distinct relative to records that not empty, haw match records contain a Latin letters, and haw many records contain a mix of Hebrew and English letters in one cell?


In [289]:

# Clean and standardize workplace names.
data["ShemAvoda"] = data["ShemAvoda"].astype("string").str.strip().replace("", pd.NA)

# Identify Hebrew and Latin characters.
has_hebrew = data["ShemAvoda"].str.contains(r"[א-ת]", na=False)
has_latin = data["ShemAvoda"].str.contains(r"[A-Za-z]", na=False)

# Separate Hebrew-only, Latin-only, and mixed-language records.
hebrew_names = data.loc[has_hebrew & ~has_latin, "ShemAvoda"]
latin_names = data.loc[has_latin & ~has_hebrew, "ShemAvoda"]
mixed_names = data.loc[has_hebrew & has_latin, "ShemAvoda"]

# Display overall statistics.
print(f"\n{' WORKPLACE NAME | ShemAvoda ':═^75}")
print(f"{'Missing / empty records':<40} {data['ShemAvoda'].isna().sum():>10,}")
print(f"{'Nonempty records':<40} {data['ShemAvoda'].count():>10,}")
print(f"{'Total unique values':<40} {data['ShemAvoda'].nunique():>10,}")

# Display Hebrew-only unique values and frequencies.
print(f"\n{' HEBREW ONLY ':═^75}")
print(f"Records: {len(hebrew_names):,} | Unique values: {hebrew_names.nunique():,}")
print(hebrew_names.value_counts().to_string() if not hebrew_names.empty else "No records found.")

# Display Latin-only unique values and frequencies.
print(f"\n{' LATIN ONLY ':═^75}")
print(f"Records: {len(latin_names):,} | Unique values: {latin_names.nunique():,}")
print(latin_names.value_counts().to_string() if not latin_names.empty else "No records found.")

# Display mixed Hebrew-Latin unique values and frequencies.
print(f"\n{' MIXED HEBREW + LATIN ':═^75}")
print(f"Records: {len(mixed_names):,} | Unique values: {mixed_names.nunique():,}")
print(mixed_names.value_counts().to_string() if not mixed_names.empty else "No records found.")
print("═" * 75)



═══════════════════════ WORKPLACE NAME | ShemAvoda ════════════════════════
Missing / empty records                           0
Nonempty records                                200
Total unique values                              82

═══════════════════════════════ HEBREW ONLY ═══════════════════════════════
Records: 186 | Unique values: 75
ShemAvoda
מסעדה                     10
משרד החינוך                9
בנק לאומי                  7
משטרה                      7
בית ספר תיכון              7
סופר פארם                  6
אינטל                      6
קניון עזריאלי              6
קוסמטיקאית עצמאית          5
למ"ס                       5
סולל בונה                  5
גידול תפוזים               4
בית מלון                   4
שופר סל                    4
בנק מזרחי                  4
בית חולים הדסה עין כרם     4
אוסם                       4
נבידיה                     4
משרד ממשלתי                4
אמדוקס                     4
חברת הייטק                 3
מקורות                     3
תיאטרון י

Conclusion: None of the 200 workplace records is empty. There are 82 distinct values among the 200 nonempty records (41%). Fourteen records contain Latin letters, and one of those also contains Hebrew letters in the same cell.

In [290]:
# Do bilingual employer names fragment the same workplace into different TF-IDF terms? Inspect normalization changes before encoding.
transformed_data = data.copy()
transformed_data["ShemAvoda"] = transform_column_with_labse(data["ShemAvoda"])

review = data.loc[shem_avoda.loc[has_latin].index, ["ID", "ShemAvoda"]].copy()
review["Transformed"] = transformed_data.loc[review.index, "ShemAvoda"]
display(review.reset_index(drop=True))

KeyError: "['ID'] not in index"

The LaBSE method replaces a full value containing Latin letters with the highest-scoring existing Hebrew-only value in the same field when cosine similarity exceeds `alpha`. Otherwise, it retains the original value.

### Numerical representation of YeshuvAvoda

Complete the locality decision established above by merging `קרית גת` with `קריית גת` and applying the one-hot encoding specified in the metadata interpretation. All 200 locality values are populated, so this field requires no null replacement. Fit the category mapping on complete-label records and retain IDs solely for alignment.

In [ ]:
# Do workplace-location spellings split equivalent locations? Standardize their representation before feature construction.
survey = transformed_data.loc[transformed_data["ID"].notna()].copy()
feature_fit_rows = survey["SemelMishlachSofi"].astype("string").str.replace(r"\.0$", "", regex=True).str.fullmatch(r"[0-9]{4}", na=False)

locality_values = survey["YeshuvAvoda"].astype("string").str.strip().map(transform_text, na_action="ignore").to_frame("YeshuvAvoda")

from sklearn.preprocessing import OneHotEncoder
locality_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
locality_encoder.fit(locality_values.loc[feature_fit_rows])
locality_matrix = locality_encoder.transform(locality_values)
locality_features = pd.DataFrame.sparse.from_spmatrix(locality_matrix, index=survey.index, columns=locality_encoder.get_feature_names_out(["YeshuvAvoda"]))
print("YeshuvAvoda one-hot matrix:", locality_matrix.shape)

### TF-IDF representation of ShemAvoda

For `ShemAvoda`, retain TF-IDF without L2 normalization because workplace names can share words while differing through additional identifying terms. With ordinary term counts, an added term contributes another weighted feature without rescaling the shared words. This preserves weighted text magnitude as potential classification information. Whether that information improves prediction must be assessed against L2-normalized TF-IDF within training-fold validation.

Lowercase the text after LaBSE, retain Hebrew and Latin words and single-letter initials, and use unsmoothed IDF. The matrix below describes the current survey. Vocabulary and IDF must be fitted within each training partition when evaluating a classifier. [scikit-learn TF-IDF documentation](https://scikit-learn.org/stable/modules/feature_extraction.html#tfidf-term-weighting)


In [ ]:
# Prepare employer names for consistent text analysis.
shem_avoda_transformed = transformed_data.loc[transformed_data["ID"].notna(), "ShemAvoda"].astype("string").str.strip().map(transform_text, na_action="ignore")

# Keep employer names in a separate vocabulary.
shem_avoda_vectorizer = TfidfVectorizer(lowercase=True, token_pattern=r"(?u)\b\w+\b", binary=False, sublinear_tf=False, smooth_idf=False, norm=None)

# Learn vocabulary and word weights from the fitting records.
shem_avoda_vectorizer.fit(shem_avoda_transformed.loc[feature_fit_rows])

# Apply the same vocabulary to all employer names.
shem_avoda_tfidf = shem_avoda_vectorizer.transform(shem_avoda_transformed)

# Identify the resulting employer features.
shem_avoda_terms = shem_avoda_vectorizer.get_feature_names_out()

# Report the number of employer features.
print("Employer name features:", shem_avoda_tfidf.shape[1])

# Report the number of nonzero feature values.
print("Nonzero feature values:", shem_avoda_tfidf.nnz)

**Conclusion:** The workplace-name column is represented numerically by TF-IDF weights after the LaBSE transformation. All 200 workplace names are populated, so this field needs no null replacement. Its vocabulary and IDF are fitted on complete-label records. Classifier validation must refit these learned quantities inside each training fold.

## From workplace normalization to occupational evidence

The assignment asks for an occupation prediction from a questionnaire that can be submitted with unanswered fields. The remaining columns describe different things: the organization, the participant's own work, employment arrangements, education, age, dates, and existing classifications. Those differences determine what can be inferred from a response and what preparation it needs.

The investigation below starts from the workplace transformation above. Each field is interpreted against its metadata and the assignment, examined using the recorded answers, and then prepared according to the resulting conclusion. The original answers remain available alongside the prepared values.

In [ ]:
# Align employer features with their questionnaire records.
shem_avoda_features = pd.DataFrame.sparse.from_spmatrix(shem_avoda_tfidf, index=survey.index, columns="ShemAvoda__" + shem_avoda_terms)

# Replace employer and locality responses with their numerical features.
prepared_data = survey.drop(columns=["ShemAvoda", "YeshuvAvoda"]).join(shem_avoda_features).join(locality_features)

# Keep the fitted transformations for processing future questionnaires.
field_encoders = {"YeshuvAvoda": locality_encoder, "ShemAvoda": shem_avoda_vectorizer}

### Analysis of SugAvoda column

The metadata defines `SugAvoda` as the workplace's main activity. This is an open description of what the organization does. Whole-answer categories would treat related phrases as unrelated labels, while TF-IDF can retain shared activity terms and the words that distinguish them.

Which recorded values describe an activity, and which provide no usable description? Inspect the answer frequencies and full values containing Latin letters before defining the treatment.

In [ ]:
# Does employer activity describe the organization rather than the respondent? Inspect its wording before using it as contextual evidence.
workplace_activity = survey["SugAvoda"].astype("string").str.strip().replace("", pd.NA)
display(workplace_activity.value_counts(dropna=False).rename_axis("Employer activity").reset_index(name="Records"))

display(workplace_activity.loc[workplace_activity.str.contains(r"[A-Za-z]", na=False)].rename("Employer activity").reset_index(drop=True).to_frame())

**Conclusion and preparation:** The 175 populated answers contain 78 distinct values, including workplace types, activities, and job descriptions. Twenty-five answers are blank. The assignment permits partially answered questionnaires, so a blank does not justify inventing an activity or deleting the record. Use empty text only for an unanswered cell.

Retain all recorded wording, including the four `חסוי` responses and the three Latin-containing values. `חסוי` records that the participant treats the workplace activity as confidential. This supplies context that an omitted answer does not, so its term remains available to the classifier. Encode this field with the same TF-IDF settings as `ShemAvoda`, using its own vocabulary. Fit vocabulary and IDF within each training fold during validation. [TF-IDF documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)

In [ ]:
# How can employer activity help classification without replacing the respondent’s own work? Build a separate text representation.
activity_text = workplace_activity.map(transform_text, na_action="ignore").fillna("")

activity_vectorizer = clone(shem_avoda_vectorizer)
activity_vectorizer.fit(activity_text.loc[feature_fit_rows])

activity_matrix = activity_vectorizer.transform(activity_text)
activity_features = pd.DataFrame.sparse.from_spmatrix(activity_matrix, index=survey.index, columns="SugAvoda__" + activity_vectorizer.get_feature_names_out())

prepared_data = prepared_data.drop(columns="SugAvoda").join(activity_features)
field_encoders["SugAvoda"] = activity_vectorizer

print("SugAvoda TF-IDF matrix:", activity_matrix.shape)

### Analysis of ShemMachlaka column

The metadata asks for the division or department in which the participant works. Department names may combine a broad function with a specialization, so individual words can carry useful information beyond the identity of the complete answer. A blank records an unanswered question and does not establish that no department exists.

Do the observed answers form a documented category set, or do they contain varied descriptions and abbreviations? Inspect their frequencies to decide how much wording the representation must retain.

In [ ]:
# Does the department name add useful context, and how often is it answered? Examine coverage and distinct values first.
department_names = survey["ShemMachlaka"].astype("string").str.strip().replace("", pd.NA)
display(department_names.value_counts(dropna=False).rename_axis("Department name").reset_index(name="Records"))

**Conclusion and preparation:** The 136 populated answers contain 61 distinct values, including department names, functions, job wording, and six Latin-containing entries. Four answers are `חסוי`, while 64 are blank. The response `חסוי` indicates that the participant treats the department information as confidential. This provides context about the department, even though it does not disclose its name, and differs from an unanswered question. Retain it as a text feature. A blank does not establish that no department exists. Use empty text only for missing answers and encode every recorded value with this field’s TF-IDF vocabulary.

In [ ]:
# Can department names contribute context independently of job descriptions? Encode them in a separate feature block.
department_text = department_names.map(transform_text, na_action="ignore").fillna("")

department_vectorizer = clone(shem_avoda_vectorizer)
department_vectorizer.fit(department_text.loc[feature_fit_rows])

department_matrix = department_vectorizer.transform(department_text)
department_features = pd.DataFrame.sparse.from_spmatrix(department_matrix, index=survey.index, columns="ShemMachlaka__" + department_vectorizer.get_feature_names_out())
prepared_data = prepared_data.drop(columns="ShemMachlaka").join(department_features)
field_encoders["ShemMachlaka"] = department_vectorizer

print("ShemMachlaka TF-IDF matrix:", department_matrix.shape)

### Analysis of SugMachlaka column

The metadata defines `SugMachlaka` as the department’s main activity. Each row is one participant’s questionnaire. The participant may write `כנ"ל`, meaning “the same,” after the preceding answer instead of repeating its text. Here the preceding field is `ShemMachlaka`, so that row’s department name supplies the repeated answer.

Inspect the recorded activities and references before completing the text transformation.

In [ ]:
# Are department-activity responses actual descriptions or references to other answers? Inspect repeated placeholders before vectorization.
department_activity = survey["SugMachlaka"].astype("string").str.strip().replace("", pd.NA)
display(department_activity.value_counts(dropna=False).rename_axis("Department activity").reset_index(name="Records"))

**Conclusion and preparation:** The field has 134 blanks and 66 populated answers, including eight `כנ"ל` references. Copy `ShemMachlaka` from the same row into those eight references because the participant has explicitly repeated the preceding answer. Keep genuinely unanswered cells empty. Then apply the existing LaBSE method to this column with its default `alpha=0.5` before TF-IDF. The method selects Hebrew values already present in this feature.

In [ ]:
# Would a reference such as “same as above” become a meaningless token? Resolve supported within-record references before encoding.
department_activity_text = department_activity.copy()
department_activity_text.loc[department_activity_text.eq('כנ"ל')] = department_names
department_activity_text = department_activity_text.map(transform_text, na_action="ignore").fillna("")

department_activity_text = transform_column_with_labse(department_activity_text)

department_activity_vectorizer = clone(shem_avoda_vectorizer)
department_activity_vectorizer.fit(department_activity_text.loc[feature_fit_rows])

department_activity_matrix = department_activity_vectorizer.transform(department_activity_text)
department_activity_features = pd.DataFrame.sparse.from_spmatrix(department_activity_matrix, index=survey.index, columns="SugMachlaka__" + department_activity_vectorizer.get_feature_names_out())
prepared_data = prepared_data.drop(columns="SugMachlaka").join(department_activity_features)
field_encoders["SugMachlaka"] = department_activity_vectorizer
print("SugMachlaka TF-IDF matrix:", department_activity_matrix.shape)

### Analysis of EzoAvoda column

The metadata asks what work the participant performs. The answer can state an occupation or describe duties. A non-disclosure response records a decision to withhold details, while a blank supplies no response. Preserve that distinction and the participant’s wording.

Inspect the descriptions, non-disclosure responses, and Latin-containing values to determine the preparation required.

In [ ]:
# Which words describe what the respondent actually does? Inspect this primary occupation signal before feature engineering.
work_descriptions = survey["EzoAvoda"].astype("string").str.strip().replace("", pd.NA)
display(work_descriptions.value_counts(dropna=False).rename_axis("Work description").reset_index(name="Records"))

**Conclusion and preparation:** There are 193 populated answers, seven blanks, and 84 distinct populated values. Two answers say `חסוי` and two say `לא יכול לפרט`. Standardize `לא יכול לפרט` to `חסוי` because both express non-disclosure. The response records that the participant treats details of the work as confidential. Retaining it gives the classifier that context while keeping it distinct from an unanswered question. The seven blanks cannot be interpreted as unemployment because the assignment permits omitted answers. Convert only missing values to empty text. Apply the existing LaBSE method with its default `alpha=0.5`, then encode the returned values with TF-IDF.

In [ ]:
# Occupation descriptions provide direct evidence of the work performed; represent them separately from employer activity.
work_description_text = work_descriptions.map(transform_text, na_action="ignore").fillna("")

work_description_text = transform_column_with_labse(work_description_text)

work_description_vectorizer = clone(shem_avoda_vectorizer)
work_description_vectorizer.fit(work_description_text.loc[feature_fit_rows])

work_description_matrix = work_description_vectorizer.transform(work_description_text)
work_description_features = pd.DataFrame.sparse.from_spmatrix(work_description_matrix, index=survey.index, columns="EzoAvoda__" + work_description_vectorizer.get_feature_names_out())
prepared_data = prepared_data.drop(columns="EzoAvoda").join(work_description_features)
field_encoders["EzoAvoda"] = work_description_vectorizer
print("EzoAvoda TF-IDF matrix:", work_description_matrix.shape)

### Analysis of TeurPeula column

The metadata asks for the main actions performed at work. Verbs and the objects of those actions distinguish duties, so retaining phrase content is more useful than assigning a separate category to every answer. Each row belongs to one participant. When the participant writes `כנ"ל`, meaning “the same,” the task answer repeats the preceding `EzoAvoda` answer in that row.

Inspect the reported tasks and reference expressions before completing the transformation.

In [ ]:
# Are task descriptions substantive or shortcuts referring to other fields? Identify references before treating them as work evidence.
task_descriptions = survey["TeurPeula"].astype("string").str.strip().replace("", pd.NA)
display(task_descriptions.value_counts(dropna=False).rename_axis("Task description").reset_index(name="Records"))

**Conclusion and preparation:** Of 64 populated answers, 12 are `כנ"ל` and 52 explicitly describe tasks. Another 136 answers are blank. An unanswered task field does not establish that the participant performs no tasks, so leave genuine blanks empty. For each `כנ"ל`, copy the resolved `EzoAvoda` text from the same row because the participant has stated that the answer is the same. Apply the existing LaBSE method to the completed task column with its default `alpha=0.5`, then perform TF-IDF.

In [ ]:
# How can task evidence be retained when respondents refer to another answer? Resolve supported same-record references before vectorization.
task_text = task_descriptions.copy()
task_text.loc[task_text.eq('כנ"ל')] = work_description_text
task_text = task_text.map(transform_text, na_action="ignore").fillna("")

task_text = transform_column_with_labse(task_text)

task_vectorizer = clone(shem_avoda_vectorizer)
task_vectorizer.fit(task_text.loc[feature_fit_rows])

task_matrix = task_vectorizer.transform(task_text)
task_features = pd.DataFrame.sparse.from_spmatrix(task_matrix, index=survey.index, columns="TeurPeula__" + task_vectorizer.get_feature_names_out())
prepared_data = prepared_data.drop(columns="TeurPeula").join(task_features)
field_encoders["TeurPeula"] = task_vectorizer
print("TeurPeula TF-IDF matrix:", task_matrix.shape)

### Analysis of TeurTafkid column

The metadata asks for a job title if one exists. An unanswered optional title does not establish that the participant lacks an occupation. Titles can contain qualifiers such as deputy or manager, so retain their full wording.

Inspect the recorded titles and reference expressions before encoding this field.

In [ ]:
# Does the job title add evidence beyond the description of actual work, and how often is it present? Inspect it separately.
job_titles = survey["TeurTafkid"].astype("string").str.strip().replace("", pd.NA)
display(job_titles.value_counts(dropna=False).rename_axis("Job title").reset_index(name="Records"))

**Conclusion and preparation:** There are 172 blanks and 28 populated answers. A blank may reflect either no formal title or an omitted answer, so neither case justifies inventing a title. Four answers are `כנ"ל` and one is `0`. Copy the resolved `TeurPeula` answer into each reference, working from left to right within that participant’s row. Thus `טכנאית רדיותרפיה | כנ"ל | כנ"ל` becomes three occurrences of `טכנאית רדיותרפיה`. Retain `0` as recorded because its meaning has not been established. Convert only missing values to empty text and encode the completed title field with TF-IDF.

In [ ]:
# Can a stated title help when present without inventing one when absent? Encode this optional field separately.
title_text = job_titles.copy()
title_text.loc[title_text.eq('כנ"ל')] = task_text

title_text = title_text.map(transform_text, na_action="ignore").fillna("")

title_vectorizer = clone(shem_avoda_vectorizer)
title_vectorizer.fit(title_text.loc[feature_fit_rows])

title_matrix = title_vectorizer.transform(title_text)
title_features = pd.DataFrame.sparse.from_spmatrix(title_matrix, index=survey.index, columns="TeurTafkid__" + title_vectorizer.get_feature_names_out())
prepared_data = prepared_data.drop(columns="TeurTafkid").join(title_features)
field_encoders["TeurTafkid"] = title_vectorizer
print("TeurTafkid TF-IDF matrix:", title_matrix.shape)

### Analysis of MenahelEtMi column

The metadata records supervisory responsibility using three codes: 1 for employees, 2 for managers, and 3 for both. Treat the responses as ordered management levels and interpret an unanswered question as no managerial responsibility. Encode this as zero and preserve levels 0–3 in one numeric feature.

Count the recorded levels and missing answers to establish how many records receive the zero encoding.

In [ ]:
# What do the management codes mean and how are they distributed? Check the metadata and responses before using responsibility as context.
management_level = survey["MenahelEtMi"]
display(management_level.value_counts(dropna=False).sort_index().rename_axis("Management response").reset_index(name="Records"))

**Conclusion and preparation:** Codes 1, 2, and 3 occur in 8, 11, and 13 records respectively. Assign zero to the 168 unanswered records and retain the recorded levels for the other 32. Zero represents the interpretation of no managerial responsibility. Keep the original responses in the source table.

In [ ]:
# How can management responsibility contribute without treating response codes as measured quantities? Encode the categories as categories.
prepared_data["MenahelEtMi"] = management_level.fillna(0).astype("int64")

### Analysis of MaamadAvoda column

The metadata defines seven employment arrangements: employee, self-employed with one or two employees, self-employed with at least three employees, self-employed without employees, cooperative member, kibbutz member, and unpaid family worker. The code numbers identify arrangements rather than measuring a quantity, so this field requires one-hot encoding.

Inspect the observed categories and unanswered records before preparing the categorical input.

In [ ]:
# Which employment statuses determine whether follow-up questions apply? Inspect their distribution before interpreting salary-source answers.
employment_status = survey["MaamadAvoda"]
display(employment_status.value_counts(dropna=False).sort_index().rename_axis("Employment status").reset_index(name="Records"))

**Conclusion and preparation:** The recorded employment-status codes identify different arrangements, not quantities. There are two unanswered records. Assign these the additional category `0`, meaning an unanswered employment-status question, and retain every recorded code. One-hot encode the categories separately so the added code does not impose an ordering or distance. Keep every category column (`drop=None`) and handle unseen categories without changing the fitted vocabulary. [scikit-learn OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).

Use the codebook categories explicitly so an allowed response retains its own column even when absent from a training fold.

In [ ]:
# Can employment arrangement add context without implying numerical order? Encode the status categories separately.
employment_categories = employment_status.fillna(0).astype("Int64").astype("string")

employment_encoder = OneHotEncoder(categories=[["0", "1", "2", "3", "4", "5", "6", "7"]], handle_unknown="ignore", sparse_output=True)
employment_encoder.fit(employment_categories.loc[feature_fit_rows].to_frame("MaamadAvoda"))

employment_matrix = employment_encoder.transform(employment_categories.to_frame("MaamadAvoda"))
employment_features = pd.DataFrame.sparse.from_spmatrix(employment_matrix, index=survey.index, columns=employment_encoder.get_feature_names_out(["MaamadAvoda"]))
prepared_data = prepared_data.drop(columns="MaamadAvoda").join(employment_features)
field_encoders["MaamadAvoda"] = employment_encoder
print("MaamadAvoda one-hot matrix:", employment_matrix.shape)

### Analysis of MakorSachar column

The metadata defines four salary sources and restricts this question to employees and cooperative members, corresponding to `MaamadAvoda` codes 1 and 5. Salary source is nominal and requires one-hot encoding. A blank can mean that the question does not apply or that an applicable answer was omitted. Its treatment therefore depends on the questionnaire's explicit routing condition.

Count salary-source responses and apply that condition to distinguish unanswered, non-applicable, and contradictory records.

In [ ]:
# Are salary-source blanks inapplicable, unanswered, or inconsistent with employment status? Check the routing logic before preparation.
salary_source = survey["MakorSachar"]
display(salary_source.value_counts(dropna=False).sort_index().rename_axis("Salary source").reset_index(name="Records"))

salary_status_known = survey["MaamadAvoda"].notna()
salary_eligible = survey["MaamadAvoda"].isin([1, 5])
salary_ineligible = salary_status_known & ~salary_eligible
display(pd.Series({"Eligible with an answer": int((salary_eligible & salary_source.notna()).sum()), "Ineligible without an answer": int((salary_ineligible & salary_source.isna()).sum()), "Ineligible with an answer": int((salary_ineligible & salary_source.notna()).sum()), "Unknown eligibility without an answer": int((~salary_status_known & salary_source.isna()).sum())}).rename_axis("Salary eligibility check").reset_index(name="Records"))

salary_eligibility_issues = survey.loc[salary_ineligible & salary_source.notna(), ["ID", "MaamadAvoda", "MakorSachar"]]
display(salary_eligibility_issues)

**Conclusion and preparation:** The field contains 186 recorded answers and 14 blanks. Use the existing category `4 — אחר` (Other) for the 14 missing responses as the selected preparation rule. This is a modeling assignment, not evidence that the participant reported another payer. Retain all populated codes, including the nine responses that contradict the employment-status routing rule, and preserve those inconsistencies in the audit table. One-hot encode the resulting nominal categories without giving their numbers a quantitative meaning. [scikit-learn OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).

In [ ]:
# Can salary source add occupational context without treating category IDs as quantities? Encode it categorically and retain the limits of the missing-value rule.
salary_categories = salary_source.fillna(4).astype("Int64").astype("string")

salary_encoder = OneHotEncoder(categories=[["1", "2", "3", "4"]], handle_unknown="ignore", sparse_output=True)
salary_encoder.fit(salary_categories.loc[feature_fit_rows].to_frame("MakorSachar"))

salary_matrix = salary_encoder.transform(salary_categories.to_frame("MakorSachar"))
salary_features = pd.DataFrame.sparse.from_spmatrix(salary_matrix, index=survey.index, columns=salary_encoder.get_feature_names_out(["MakorSachar"]))
prepared_data = prepared_data.drop(columns="MakorSachar").join(salary_features)
field_encoders["MakorSachar"] = salary_encoder
print("MakorSachar one-hot matrix:", salary_matrix.shape)

### Analysis of TeudaGvoha column

The metadata records the highest certificate or degree. Codes 1–7 cover school qualifications through a doctorate, while 8 means another certificate and 9 means no certificate. The entire code list therefore cannot be treated as one increasing numeric measure. Use one-hot encoding to preserve the qualification categories.

Inspect their frequencies and the number of unanswered qualifications before constructing the categorical input.

In [ ]:
# Which qualifications were reported and which responses are absent? Inspect the field before deciding how to represent missingness.
qualifications = survey["TeudaGvoha"]
display(qualifications.value_counts(dropna=False).sort_index().rename_axis("Qualification").reset_index(name="Records"))

**Conclusion and preparation:** The field contains 198 recorded qualifications and two blanks. Codes 1–7 identify specific certificates, code 8 covers any other certificate, and code 9 denotes no certificate. For this preparation, assign the two missing qualification responses to code 9, interpreting the omission as no certificate reported. This is an explicit interpretation of nonresponse. All populated answers remain unchanged. One-hot encode the categories so their numerical labels do not impose an ordering. [scikit-learn OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).

Use the codebook categories explicitly so an allowed response retains its own column even when absent from a training fold.

In [ ]:
# Can qualifications help distinguish occupations without assuming an unanswered field means no qualification? Encode categories while identifying the substitution assumption.
qualification_categories = qualifications.fillna(9).astype("Int64").astype("string")

qualification_encoder = OneHotEncoder(categories=[["1", "2", "3", "4", "5", "6", "7", "8", "9"]], handle_unknown="ignore", sparse_output=True)
qualification_encoder.fit(qualification_categories.loc[feature_fit_rows].to_frame("TeudaGvoha"))

qualification_matrix = qualification_encoder.transform(qualification_categories.to_frame("TeudaGvoha"))
qualification_features = pd.DataFrame.sparse.from_spmatrix(qualification_matrix, index=survey.index, columns=qualification_encoder.get_feature_names_out(["TeudaGvoha"]))
prepared_data = prepared_data.drop(columns="TeudaGvoha").join(qualification_features)
field_encoders["TeudaGvoha"] = qualification_encoder
print("TeudaGvoha one-hot matrix:", qualification_matrix.shape)

### Analysis of shnotlimud column

The metadata defines years of schooling as an integer measurement. The preparation specification sets an admissible range of 8–80 years and converts values outside that range to missing. Apply this explicit rule to the recorded measurement, then learn any replacement value from training records.

Inspect the observed distribution to identify the entries affected by the specified range.

In [ ]:
# Are schooling years plausible, missing, or unusually low? Inspect the observed distribution before applying numerical bounds.
school_years = survey["shnotlimud"]
display(school_years.value_counts(dropna=False).sort_index().rename_axis("Schooling years").reset_index(name="Records"))

**Conclusion and preparation:** There are 198 recorded values between 7 and 22 years and two blank answers. The two seven-year values fall below the specified 8–80 preparation range and become missing, giving four missing measurements. Zero would assert no schooling and would violate that range. Use the median of admissible complete-label training records as a central replacement without using the occupation label to estimate schooling. Fit the replacement within each training fold during validation. Preserve the reported values in the source data. [Median imputation](https://scikit-learn.org/stable/modules/impute.html#univariate-feature-imputation)

In [ ]:
# How can models receive a complete schooling feature without learning an imputation value from evaluation records? Fit the median on the designated fitting rows.
schooling_values = school_years.where(school_years.between(8, 80)).to_frame("shnotlimud")

from sklearn.impute import SimpleImputer
schooling_imputer = SimpleImputer(strategy="median")
schooling_imputer.fit(schooling_values.loc[feature_fit_rows])

prepared_data["shnotlimud"] = schooling_imputer.transform(schooling_values).ravel()
field_encoders["shnotlimud"] = schooling_imputer
print("Schooling values replaced:", int(schooling_values["shnotlimud"].isna().sum()))
print("Training median:", schooling_imputer.statistics_[0])

### Analysis of Gil column

The metadata defines age as an integer measurement. The preparation specification admits ages from 18 to 120 and converts values outside that range to missing. Age remains numeric because differences between recorded ages have a quantitative meaning.

Inspect its distribution to determine whether the rule removes any observed measurements and how many replacements will be needed.

In [ ]:
# Does age provide useful context, and which values are missing or outside the selected range? Inspect before numerical preparation.
age = survey["Gil"]
display(age.value_counts(dropna=False).sort_index().rename_axis("Age").reset_index(name="Records"))

**Conclusion and preparation:** All 198 recorded ages, ranging from 22 to 78, satisfy the specified 18–120 range. Two answers are blank. Zero is not a plausible age under this specification, and dropping those records would discard their remaining information. Replace the two blanks with the median age fitted on complete-label training records. This supplies a central numeric input without claiming to recover the participant's actual age. Refit the replacement within each training fold during validation.

In [ ]:
# How can missing ages be represented consistently without fitting on evaluation data? Use the designated fitting population for imputation.
age_values = age.where(age.between(18, 120)).to_frame("Gil")

age_imputer = SimpleImputer(strategy="median")
age_imputer.fit(age_values.loc[feature_fit_rows])

prepared_data["Gil"] = age_imputer.transform(age_values).ravel()
field_encoders["Gil"] = age_imputer
print("Ages replaced:", int(age_values["Gil"].isna().sum()))
print("Training median:", age_imputer.statistics_[0])

### Preparation of TarichHathala column

The metadata identifies the date the participant started the current role. The useful measurement is years on the job. Under the assignment's immediate-processing interpretation, tenure runs from that starting date to the processing date. Because no survey or last-update date is supplied, use the current execution date as that reference.

Record the reference date so the calculation is reproducible and state this substitution in the presentation. The model receives elapsed years rather than the calendar date.

In [ ]:
# What does the role-start date tell us about experience, and what reference date is available? Compute elapsed time while distinguishing it from survey-time tenure.
start_dates = pd.to_datetime(survey["TarichHathala"])

reference_date = pd.Timestamp.now(tz="UTC").normalize().tz_localize(None)
years_on_job = (reference_date - start_dates).dt.days / 365.25

print("Tenure reference date:", reference_date.date())
display(years_on_job.agg(["count", "min", "max", "median"]).rename_axis("Statistic").reset_index(name="Years on the job").round(2))

**Conclusion and preparation:** All 200 records yield a tenure value. Convert elapsed days to years using 365.25 days per year and retain the resulting continuous measurement. In the prepared table, `TarichHathala` contains years on the job, while the original table retains the starting date. The execution date substitutes for the unavailable survey date and must accompany the reported results.

In [ ]:
# How can duration values remain comparable within this run? Record the single reference date used for every conversion.
prepared_data["TarichHathala"] = years_on_job

tenure_reference_date = reference_date.date().isoformat()

### Analysis of SemelAnafSofi column

The metadata defines a four-character industry classification. Its digits identify categories in a hierarchy rather than a numeric magnitude, and `X` denotes incomplete coding. Leading zeros therefore belong to the code representation.

Inspect the short stored values against Excel's number format before restoring a zero. Preserve partial codes. The preparation specification permits this field as a predictor only when the code is available at prediction time, which the supplied assignment does not establish.

In [ ]:
# Are industry codes categorical identifiers and are any incomplete? Inspect their format and coverage before encoding.
industry_codes = survey["SemelAnafSofi"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)

from openpyxl import load_workbook
source_workbook = load_workbook(io.BytesIO(raw), read_only=True, data_only=True)
source_sheet = source_workbook["Survey Dataset"]
industry_position = list(next(source_sheet.values)).index("SemelAnafSofi") + 1
short_industry_codes = industry_codes.str.fullmatch(r"[0-9]{1,3}", na=False)
industry_storage = survey.loc[short_industry_codes, ["ID", "SemelAnafSofi"]].copy()
industry_storage["Excel format"] = [source_sheet.cell(int(index) + 2, industry_position).number_format for index in industry_storage.index]
industry_storage["ID"] = industry_storage["ID"].astype("Int64")
display(industry_storage.rename(columns={"SemelAnafSofi": "Industry code"}).reset_index(drop=True))
source_workbook.close()

display(industry_codes.loc[industry_codes.str.contains("X", na=False)].value_counts().rename_axis("Partial industry code").reset_index(name="Records"))

**Conclusion and preparation:** Five numeric values use Excel's `0000` format and require a leading zero to reproduce their four-position codes. Nine records contain `X`: six `XXXX` and one each of `851X`, `852X`, and `3XXX`. These are partial classifications, not empty cells, so preserve every known and unresolved position. Restore leading zeros and one-hot encode the resulting categorical values. Keep the industry matrix separate from the primary predictors because its availability before occupation prediction is not established.

In [ ]:
# Does industry classification add independent context, and will it be available for new cases? Prepare it separately pending that decision.
industry_values = industry_codes.where(~industry_codes.str.fullmatch(r"[0-9]{1,4}", na=False), industry_codes.str.zfill(4)).to_frame("SemelAnafSofi")

industry_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
industry_encoder.fit(industry_values.loc[feature_fit_rows])
industry_matrix = industry_encoder.transform(industry_values)
industry_features = pd.DataFrame.sparse.from_spmatrix(industry_matrix, index=survey.index, columns=industry_encoder.get_feature_names_out(["SemelAnafSofi"]))

prepared_data = prepared_data.drop(columns="SemelAnafSofi")
field_encoders["SemelAnafSofi"] = industry_encoder
print("SemelAnafSofi one-hot matrix:", industry_matrix.shape)

### Analysis of SemelMishlachSofi column

The assignment identifies this four-character occupation code as the target. Codes containing `X` were not fully classified by the professional coders. They cannot supply complete labels for supervised training or full-code accuracy, but their known digits must be retained alongside later predictions.

Count complete labels, partial patterns, and records per complete class to establish the training population and its class support.

In [ ]:
# How many occupation codes are fully known, and are classes frequent enough to evaluate? Count complete labels and rare classes first.
occupation_codes = survey["SemelMishlachSofi"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
occupation_codes = occupation_codes.where(~occupation_codes.str.fullmatch(r"[0-9]{1,4}", na=False), occupation_codes.str.zfill(4))

complete_target = occupation_codes.str.fullmatch(r"[0-9]{4}", na=False)
class_counts = occupation_codes.loc[complete_target].value_counts()
display(class_counts.rename_axis("Occupation code").reset_index(name="Records"))
display(occupation_codes.loc[~complete_target].value_counts().rename_axis("Partial occupation code").reset_index(name="Records"))
display(pd.Series({"Complete labels": int(complete_target.sum()), "Incomplete labels": int((~complete_target).sum()), "Complete classes": class_counts.size, "Singleton classes": int(class_counts.eq(1).sum())}).rename_axis("Label summary").reset_index(name="Count"))

**Conclusion and preparation:** The 191 complete labels cover 59 classes, including 24 represented by one record. Nine records have partial labels: four `1XXX`, four `XXXX`, and one `335X`. Use complete codes as categorical targets and reserve the nine partial records for prediction. Preserve known prefixes for comparison with the eventual predicted codes. Singleton classes constrain validation coverage and must remain visible in evaluation reporting.

In [ ]:
# Which occupation digits are known versus unresolved? Use complete codes for supervised training and preserve partial codes for consistency checks.
prepared_data["SemelMishlachSofi"] = occupation_codes
training_ids = survey.loc[complete_target, "ID"].astype("int64")
prediction_ids = survey.loc[~complete_target, "ID"].astype("int64")
y_train = occupation_codes.loc[complete_target].copy()
partial_targets = occupation_codes.loc[~complete_target].copy()

known_prefixes = partial_targets.str.split("X").str[0]
display(pd.DataFrame({"ID": prediction_ids, "Recorded target": partial_targets, "Known prefix": known_prefixes}))

## Do feature values stay within one code digit or cross between digits?

For `SemelAnafSofi` and `SemelMishlachSofi`, examine each of the four digit positions separately. At a given position, a recurring feature value is **exclusive** when all of its observed occurrences have the same digit. It **crosses digits** when it occurs with more than one digit. Each heatmap cell shows the percentage of distinct recurring values observed in that digit group that are exclusive to it. Darker color means greater separation. Values observed only once are reported separately because one observation cannot demonstrate a repeatable relationship. A grey cell has no recurring values to assess.

The analysis compares the 16 other survey fields, including the other code. It uses recorded values after the dictionary substitutions and reference expansion, before learned encodings. Apply the field-specific missing-value rules for management, employment status, salary source, and qualification before comparison. Other missing responses supply no value for this comparison. ID is excluded because it identifies records rather than describing a participant’s work. Numeric values are compared exactly, so these plots test repeated-value separation, not a linear association.

A fixed group split reserves the final test before this investigation. The plots use development records so any resulting feature decision can be evaluated without consulting held-out labels. Incomplete occupation labels remain prediction records. Within development, an unknown `X` digit is excluded only from that position’s analysis. Leading zeros remain part of the code.

A positional digit is not a complete category definition. Occupation prefixes of one through four digits form the ISCO hierarchy. The Israeli industry classification uses a lettered section and numeric levels of two, three, and four digits, so its first numeric digit must not be called a formal section. The assignment does not specify a classification edition, and this investigation does not assign unverified code titles. [ILO occupation hierarchy](https://isco.ilo.org/en/isco-08/) and [Israel Central Bureau of Statistics industry structure](https://www.cbs.gov.il/he/mediarelease/DocLib/2012/300/29_12_300b.pdf).

In [ ]:
# Repeated text profiles can inflate random-split scores; keep identical profiles together when evaluating new wording.
from sklearn.model_selection import GroupShuffleSplit

CV_FOLDS = 3
TEST_SIZE = 0.22

text_fields = ["ShemAvoda", "SugAvoda", "ShemMachlaka", "SugMachlaka", "EzoAvoda", "TeurPeula", "TeurTafkid"]
labse_fields = ["ShemAvoda", "SugMachlaka", "EzoAvoda", "TeurPeula"]
reference_fields = {"SugMachlaka": "ShemMachlaka", "TeurPeula": "EzoAvoda", "TeurTafkid": "TeurPeula"}
raw_records = data.loc[data["ID"].notna()].copy()
raw_predictors = raw_records.drop(columns=["ID", "SemelAnafSofi", "SemelMishlachSofi"])
profile_text = raw_records[text_fields].astype("string").apply(lambda column: column.str.strip().fillna(""))
profile_keys = profile_text.apply(tuple, axis=1)
profile_groups = pd.Series(pd.factorize(profile_keys)[0], index=raw_records.index)

labelled_predictors = raw_predictors.loc[complete_target]
labelled_targets = occupation_codes.loc[complete_target]
labelled_groups = profile_groups.loc[complete_target]
development_positions, test_positions = next(GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED).split(labelled_predictors, labelled_targets, labelled_groups))
development_predictors = labelled_predictors.iloc[development_positions]
development_targets = labelled_targets.iloc[development_positions]
development_groups = labelled_groups.iloc[development_positions]
test_predictors = labelled_predictors.iloc[test_positions]
test_targets = labelled_targets.iloc[test_positions]
validation_splits = list(GroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED).split(development_predictors, development_targets, development_groups))

results_directory = Path("classification_results")
results_directory.mkdir(exist_ok=True)
split_assignments = raw_records[["ID"]].copy()
split_assignments["Group"] = profile_groups
split_assignments["Population"] = "Incomplete label"
split_assignments.loc[development_predictors.index, "Population"] = "Development"
split_assignments.loc[test_predictors.index, "Population"] = "Held-out test"
split_assignments.to_csv(results_directory / "split_assignments.csv", index=False, encoding="utf-8-sig")
display(pd.DataFrame({"Records": [len(development_targets), len(test_targets), int((~complete_target).sum())], "Complete classes": [development_targets.nunique(), test_targets.nunique(), 0]}, index=["Development", "Held-out test", "Incomplete labels"]))

In [ ]:
# Can identical descriptions have different occupation labels? Inspect conflicts before attributing every disagreement to model failure.
analysis_values = raw_records.loc[development_predictors.index].drop(columns="ID").copy()
for field in ["YeshuvAvoda"] + text_fields:
    analysis_values[field] = analysis_values[field].astype("string").str.strip().map(transform_text, na_action="ignore").replace("", pd.NA)
for field, previous_field in reference_fields.items():
    analysis_values.loc[analysis_values[field].eq('כנ"ל').fillna(False), field] = analysis_values[previous_field]
analysis_values["SemelAnafSofi"] = industry_values.loc[analysis_values.index, "SemelAnafSofi"]
analysis_values["SemelMishlachSofi"] = occupation_codes.loc[analysis_values.index]
analysis_values["TarichHathala"] = years_on_job.loc[analysis_values.index]
analysis_values["MenahelEtMi"] = analysis_values["MenahelEtMi"].fillna(0).astype("int64")
analysis_values["MaamadAvoda"] = analysis_values["MaamadAvoda"].fillna(0).astype("int64")
analysis_values["MakorSachar"] = analysis_values["MakorSachar"].fillna(4).astype("int64")
analysis_values["TeudaGvoha"] = analysis_values["TeudaGvoha"].fillna(9).astype("int64")

consistency_rows = []
value_relationships = []
for code_field in ["SemelAnafSofi", "SemelMishlachSofi"]:
    for position in range(1, 5):
        digit = analysis_values[code_field].str[position - 1]
        known_digit = digit.str.fullmatch(r"[0-9]", na=False)
        digit_values = digit.loc[known_digit].astype(int)
        digit_range = range(int(digit_values.min()), int(digit_values.max()) + 1)
        for field in analysis_values.columns.drop(code_field):
            pairs = pd.DataFrame({"Digit": digit_values, "Value": analysis_values.loc[known_digit, field].astype("string")}).dropna()
            counts = pd.crosstab(pairs["Value"], pairs["Digit"]).reindex(columns=digit_range, fill_value=0)
            totals = counts.sum(axis=1)
            distinct_digits = counts.gt(0).sum(axis=1)
            recurring = totals.ge(2)
            for value in counts.index:
                observed = counts.loc[value]
                value_relationships.append({"Code field": code_field, "Position": position, "Feature": field, "Value": value, "Occurrences": int(totals.loc[value]), "Digits": ",".join(map(str, observed.index[observed.gt(0)])), "Digit count": int(distinct_digits.loc[value]), "Recurring": bool(recurring.loc[value]), "Exclusive": bool(recurring.loc[value] and distinct_digits.loc[value] == 1)})
            for number in digit_range:
                present = counts[number].gt(0)
                repeated_values = int((present & recurring).sum())
                exclusive_values = int((present & recurring & distinct_digits.eq(1)).sum())
                percentage = 100 * exclusive_values / repeated_values if repeated_values else np.nan
                consistency_rows.append({"Code field": code_field, "Position": position, "Digit": number, "Feature": field, "Group records": int(digit_values.eq(number).sum()), "Observed records": int(counts[number].sum()), "Distinct values": int(present.sum()), "Singleton values": int((present & ~recurring).sum()), "Recurring values": repeated_values, "Exclusive recurring values": exclusive_values, "Exclusive percent": percentage})
consistency = pd.DataFrame(consistency_rows)
value_relationships = pd.DataFrame(value_relationships)
output = Path("data_preparation_results")
output.mkdir(exist_ok=True)
consistency.to_csv(output / "digit_consistency.csv", index=False, encoding="utf-8-sig")
value_relationships.to_csv(output / "digit_value_relationships.csv", index=False, encoding="utf-8-sig")

from matplotlib.colors import LinearSegmentedColormap
from IPython.display import Image
plot_directory = Path("plots")
plot_directory.mkdir(exist_ok=True)
consistency_colors = LinearSegmentedColormap.from_list("survey_consistency", ["#F8F7F1", "#A8BCA8", "#234737"])
consistency_colors.set_bad("#DCE0D4")
for code_field in ["SemelAnafSofi", "SemelMishlachSofi"]:
    figure, axes = plt.subplots(4, 1, figsize=(18, 22), layout="constrained")
    for position, axis in enumerate(axes, start=1):
        rows = consistency.loc[consistency["Code field"].eq(code_field) & consistency["Position"].eq(position)]
        matrix = rows.pivot(index="Digit", columns="Feature", values="Exclusive percent").reindex(columns=analysis_values.columns.drop(code_field))
        group_sizes = rows.groupby("Digit")["Group records"].first()
        image = axis.imshow(matrix.to_numpy(), cmap=consistency_colors, vmin=0, vmax=100, aspect="auto")
        axis.set_xticks(range(len(matrix.columns)), matrix.columns, rotation=35, ha="right", fontsize=9)
        axis.set_yticks(range(len(matrix)), [f"{digit}  (n={group_sizes.loc[digit]})" for digit in matrix.index])
        axis.set_title(f"{code_field} | digit position {position}", loc="left", color="#253D31", fontweight="bold")
        axis.set_ylabel("Digit and group size")
        for row in range(len(matrix)):
            for column in range(len(matrix.columns)):
                value = matrix.iat[row, column]
                label = f"{value:.0f}%" if pd.notna(value) else "—"
                axis.text(column, row, label, ha="center", va="center", fontsize=8, color="white" if pd.notna(value) and value >= 60 else "#253D31")
    figure.colorbar(image, ax=axes, label="Recurring values confined to this digit (%)", shrink=0.65)
    figure.suptitle("Repeated feature values: separation versus crossing between digits", fontsize=16, color="#253D31")
    figure.savefig(plot_directory / f"{code_field}_digit_consistency.png", dpi=140, facecolor="#F8F7F1")
    display(Image(filename=str(plot_directory / f"{code_field}_digit_consistency.png")))
    plt.close(figure)

supported_consistency = consistency.loc[consistency["Recurring values"].ge(3)].sort_values(["Exclusive percent", "Recurring values"], ascending=[False, False])
display(supported_consistency.groupby(["Code field", "Position"], sort=False).head(5))
crossing_examples = value_relationships.loc[value_relationships["Recurring"] & value_relationships["Digit count"].gt(1)].sort_values(["Digit count", "Occurrences"], ascending=[False, False])
display(crossing_examples.groupby(["Code field", "Position"], sort=False).head(5))

### One feature value → one target: four-digit relationship

The existing heatmaps assess **each digit position separately**. A stronger question is whether a particular observed feature **value** identifies exactly one complete four-digit occupation code, rather than merely one digit at a given position. Evaluate both relationships on development records only, with support of at least three records. A relationship is *observationally exclusive*, not a guaranteed prediction rule; coverage and contradictory values must be shown alongside 100% consistency. Do not use the occupation target as an input feature.


In [ ]:
# Identify feature values that consistently identify all four occupation digits, not just one position.
occupation_target = analysis_values["SemelMishlachSofi"].astype("string")
four_digit_rows = []
for field in analysis_values.columns.drop(["SemelMishlachSofi", "SemelAnafSofi"], errors="ignore"):
    pairs = pd.DataFrame({"Value": analysis_values[field].astype("string"), "Target": occupation_target}).dropna()
    pairs = pairs.loc[pairs["Value"].str.strip().ne("") & pairs["Target"].str.fullmatch(r"[0-9]{4}", na=False)]
    for value, group in pairs.groupby("Value", sort=False):
        counts = group["Target"].value_counts()
        dominant_code = counts.index[0]
        four_digit_rows.append({"Feature": field, "Value": value, "Support": len(group), "Distinct targets": len(counts), "Dominant target": dominant_code, "Consistency": counts.iloc[0] / len(group), "Exclusive 4 digits": len(counts) == 1 and len(group) >= 3})
four_digit_relationships = pd.DataFrame(four_digit_rows)
four_digit_relationships.to_csv(output / "four_digit_feature_relationships.csv", index=False, encoding="utf-8-sig")
supported_relationships = four_digit_relationships.loc[four_digit_relationships["Support"].ge(3)].copy()
feature_relationship_summary = supported_relationships.groupby("Feature").agg(Supported_values=("Value", "size"), Exclusive_values=("Exclusive 4 digits", "sum"), Supporting_records=("Support", "sum")).reset_index()
feature_relationship_summary["Exclusive share"] = feature_relationship_summary["Exclusive_values"] / feature_relationship_summary["Supported_values"]
feature_relationship_summary = feature_relationship_summary.sort_values(["Exclusive share", "Supporting_records"], ascending=False)
display(feature_relationship_summary.style.hide(axis="index").format({"Exclusive share": "{:.1%}"}))
display(supported_relationships.sort_values(["Exclusive 4 digits", "Support"], ascending=False).head(30).style.hide(axis="index").format({"Consistency": "{:.1%}"}))
figure, axis = plt.subplots(figsize=(11, 6), layout="constrained")
plot_rows = feature_relationship_summary.sort_values("Exclusive share")
bars = axis.barh(plot_rows["Feature"], plot_rows["Exclusive share"], color="#386A78")
axis.bar_label(bars, labels=[f'{row["Exclusive_values"]}/{row["Supported_values"]}' for _, row in plot_rows.iterrows()], padding=3)
axis.set_xlim(0, 1.16)
axis.set_xlabel("Share of recurring values identifying one observed four-digit code")
axis.set_title("One feature value → one four-digit occupation code", loc="left")
axis.spines[["top", "right"]].set_visible(False)
figure.savefig(plot_directory / "four_digit_feature_exclusivity.png", dpi=160, facecolor="#F8F7F1")
display(Image(filename=str(plot_directory / "four_digit_feature_exclusivity.png")))
plt.close(figure)


**Conclusion and preparation:** Work-description values show substantially more separation between occupation digits than the coded employment or salary categories. For example, among development records with first occupation digit `2`, all 14 recurring `EzoAvoda` values are exclusive to that digit. The same holds for all nine recurring work descriptions under first digit `5`. These are observed repeated-value relationships, not proof that a value determines an occupation in new surveys. Several 100% cells have few supported values, which is why the denominators and singleton counts accompany the plots.

The fields still provide different information. Employment and salary categories cross occupation digits, so they cannot replace the text descriptions, but crossing alone does not establish that they are useless in combination with other predictors. Keep their field-specific encodings. No new predictor is created from occupation digits because those digits belong to the target. Industry digits remain descriptive analysis until industry-code availability at prediction is established. The evidence supports retaining the separate text fields and reporting occupation accuracy at progressively longer code prefixes.

## Completed feature preparation

Each questionnaire field has already been investigated and transformed **once, in its own section**. The matrices below assemble those completed columns without reprocessing raw responses. Industry features remain separate, and the occupation code is the target, not a predictor.

The following model comparison reuses these matrices. Since learned TF-IDF and imputation parameters were fitted before the train/test split, its scores are exploratory; they are not leakage-free estimates of generalization.


In [ ]:
# Are all feature blocks and target labels aligned to the same records? Assemble them with consistent row indices before training.
from scipy import sparse
predictor_fields = prepared_data.columns.drop(["ID", "SemelMishlachSofi"]).tolist()
X_train = prepared_data.loc[complete_target, predictor_fields].copy()
X_prediction = prepared_data.loc[~complete_target, predictor_fields].copy()

training_matrix = sparse.csr_matrix(X_train.astype(pd.SparseDtype("float64", 0)).sparse.to_coo())
prediction_matrix = sparse.csr_matrix(X_prediction.astype(pd.SparseDtype("float64", 0)).sparse.to_coo())
feature_names = np.asarray(predictor_fields)
industry_training_matrix = industry_matrix[complete_target.to_numpy()]
industry_prediction_matrix = industry_matrix[(~complete_target).to_numpy()]

display(pd.DataFrame({"Records": [len(X_train), len(X_prediction)], "Numerical features": [training_matrix.shape[1], prediction_matrix.shape[1]]}, index=["Complete target", "Incomplete target"]))

### Save the completed preparation

Save the numerical inputs, fitted field encoders, original workplace transformation output, and salary-routing discrepancies. The reference date accompanies the tenure calculation. These artifacts preserve completed preparation while classifier and representation selection remain a separate evaluation step.

In [ ]:
# Can the analysis be reproduced and its assumptions inspected? Save the prepared data and feature decisions.
import joblib
import sklearn
prepared_data.to_csv(output / "prepared_survey.csv", index=False, encoding="utf-8-sig")
pd.concat([training_ids.rename("ID"), X_train, y_train.rename("SemelMishlachSofi")], axis=1).to_csv(output / "training_inputs.csv", index=False, encoding="utf-8-sig")
pd.concat([prediction_ids.rename("ID"), X_prediction, partial_targets.rename("Recorded_target"), known_prefixes.rename("Known_prefix")], axis=1).to_csv(output / "prediction_inputs.csv", index=False, encoding="utf-8-sig")
sparse.save_npz(output / "training_features.npz", training_matrix)
sparse.save_npz(output / "prediction_features.npz", prediction_matrix)
sparse.save_npz(output / "industry_training_features.npz", industry_training_matrix)
sparse.save_npz(output / "industry_prediction_features.npz", industry_prediction_matrix)

pd.Series(feature_names, name="Feature").to_csv(output / "feature_names.csv", index=False, encoding="utf-8-sig")
pd.Series(industry_encoder.get_feature_names_out(["SemelAnafSofi"]), name="Feature").to_csv(output / "industry_feature_names.csv", index=False, encoding="utf-8-sig")
salary_eligibility_issues.to_csv(output / "preparation_audit.csv", index=False, encoding="utf-8-sig")
review.to_csv(output / "workplace_transformations.csv", index=False, encoding="utf-8-sig")
joblib.dump(field_encoders, output / "feature_encoders.joblib")

manifest = {"records": len(survey), "training_records": len(X_train), "prediction_records": len(X_prediction), "features": len(feature_names), "training_ids": training_ids.tolist(), "prediction_ids": prediction_ids.tolist(), "tfidf_normalization": shem_avoda_vectorizer.norm, "tfidf_smooth_idf": False, "tenure_reference_date": tenure_reference_date, "tenure_definition": "Elapsed days from role start to the current processing date divided by 365.25.", "tenure_assumption": "Immediate processing. The current date substitutes for the unavailable survey or last-update date.", "management_missing_value": 0, "employment_missing_category": 0, "salary_missing_category": 4, "qualification_missing_category": 9, "qualification_missing_assumption": "The participant left the self-completed question blank because there was no certificate to report.", "schooling_range": [8, 80], "age_range": [18, 120], "added_missingness_indicators": False, "industry_in_primary_model": False, "classifier_trained": False, "sklearn_version": sklearn.__version__, "pandas_version": pd.__version__}
(output / "preparation_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Prepared numerical inputs and field encoders saved.")

## Classification using the existing prepared columns

No second text cleaning, LaBSE matching, TF-IDF fitting, category encoding, imputation, or clustering occurs below. The only operations after feature preparation are selecting the already prepared rows, fitting classifiers, evaluating predictions, and saving results. The three grouped folds compare classifiers using the same feature matrix.


In [ ]:
# Reuse the numerical features already prepared, without rebuilding questionnaire columns.
import time
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV, ParameterGrid
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

from sklearn.ensemble import HistGradientBoostingClassifier, VotingClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import FunctionTransformer
from lightgbm import LGBMClassifier

from sklearn.base import BaseEstimator, ClassifierMixin

def to_dense(matrix):
    return matrix.toarray() if sparse.issparse(matrix) else matrix


### Align the existing feature rows with the group split

The earlier split uses positions within the **complete-label records**. Apply those positions to the completed `training_matrix`; never use them to index the original 200-row table directly.


In [ ]:
# Select development and held-out rows from the already prepared features.
development_matrix = training_matrix[development_positions]
test_matrix = training_matrix[test_positions]
assert development_matrix.shape[0] == len(development_targets) and test_matrix.shape[0] == len(test_targets)
assert development_matrix.shape[1] == test_matrix.shape[1] == prediction_matrix.shape[1]


### Stage 1 — Define the dataset representations

Prepare **eight dataset specifications** from the existing feature columns: L2 normalization on/off, target-digit-exclusive feature selection on/off, and SVD on/off. Each specification is applied to the same questionnaire-derived matrix. The representation is fitted **within each training fold**: target-derived selection must not see validation labels, and SVD must not learn from validation rows. The eight options are compared before interpreting classifier performance, but are not selected using the held-out test.

### Stage 2 — Evaluate every compatible classifier on every dataset

Every dataset is paired with the retained classifier families and their parameter choices. Complement Naive Bayes is excluded from SVD datasets because SVD can create negative components. The results preserve dataset identity, classifier, parameter values, three fold scores, and the overall ranking. The historical experiment is preserved separately.


In [ ]:
# Combine complementary classifiers to test whether their predictions improve occupation classification.
def build_ensemble():
    return VotingClassifier(estimators=[("logistic", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)), ("forest", RandomForestClassifier(n_estimators=80, class_weight="balanced", random_state=SEED, n_jobs=1)), ("xgboost", XGBClassifier(n_estimators=40, max_depth=2, learning_rate=0.1, eval_metric="mlogloss", tree_method="hist", random_state=SEED, n_jobs=1))], voting="soft", n_jobs=1)


# Compare eight classifiers with different learning approaches on the same prepared data.
model_families = {
    "Linear SVC": (LinearSVC(class_weight="balanced", random_state=SEED), {"classifier__C": [0.1, 1]}),

    "Logistic regression": (LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED), {"classifier__C": [0.1, 1]}),

    "RBF SVC": (SVC(C=1, class_weight="balanced", random_state=SEED), {}),

    "Complement NB": (ComplementNB(alpha=1), {}),

    "XGBoost": (VotingClassifier(estimators=[("xgboost", XGBClassifier(n_estimators=40, max_depth=2, learning_rate=0.1, eval_metric="mlogloss", tree_method="hist", random_state=SEED, n_jobs=1))], voting="soft"), {}),

    "LightGBM": (LGBMClassifier(n_estimators=40, num_leaves=7, min_child_samples=2, learning_rate=0.1, verbosity=-1, random_state=SEED, n_jobs=1), {}),

    "Random forest": (RandomForestClassifier(n_estimators=80, class_weight="balanced", random_state=SEED, n_jobs=1), {"classifier__min_samples_leaf": [1, 2]}),

    "Soft voting": (build_ensemble(), {})
}


# Identify features associated with one occupation-code digit using training records only.
class DigitExclusiveSelector(TransformerMixin, BaseEstimator):

    def __init__(self, min_support=3):
        self.min_support = min_support

    def fit(self, X, y):
        values = sparse.csc_matrix(X)
        targets = np.asarray(y, dtype=str)
        digit_codes = np.asarray([[code[position] for position in range(4)] for code in targets])

        # Count the records containing each feature.
        self.support_ = np.diff(values.indptr)
        self.exclusive_ = np.zeros(values.shape[1], dtype=bool)

        # Identify features associated with a single digit at any occupation-code position.
        for column in np.flatnonzero(self.support_ >= self.min_support):
            rows = values.indices[values.indptr[column]:values.indptr[column + 1]]
            self.exclusive_[column] = any(np.unique(digit_codes[rows, position]).size == 1 for position in range(4))

        # Keep all features when none satisfy the selection rule.
        self.mask_ = self.exclusive_.copy()

        if not self.mask_.any():
            self.mask_[:] = True

        return self

    # Apply the selected features without using validation labels.
    def transform(self, X):
        check_is_fitted(self, "mask_")
        return X[:, self.mask_]


# Compare eight representations using normalization, digit selection, and dimensionality reduction.
from itertools import product

dataset_options = pd.DataFrame([{"Dataset": f"D{i:02d}", "L2": l2, "Digit selection": select, "SVD": svd} for i, (l2, select, svd) in enumerate(product([False, True], repeat=3), start=1)])

# Display the representations included in the experiment.
display(dataset_options.style.hide(axis="index"))

# Save the representation definitions for interpreting results.
dataset_options.to_csv(results_directory / "dataset_options.csv", index=False)


# Compare each classifier across compatible dataset representations.
parameter_grids = []
candidate_descriptions = []

for dataset in dataset_options.to_dict("records"):

    for family, (classifier, parameters) in model_families.items():

        # Complement NB requires nonnegative features, which SVD cannot guarantee.
        if dataset["SVD"] and family == "Complement NB":
            continue

        # Learn feature selection and dimensionality reduction within each training fold.
        parameter_grids.append({"normalize": [Normalizer() if dataset["L2"] else "passthrough"], "select": [DigitExclusiveSelector(min_support=3) if dataset["Digit selection"] else "passthrough"], "svd": [TruncatedSVD(n_components=16, random_state=SEED) if dataset["SVD"] else "passthrough"], "classifier": [classifier], **parameters})

        # Record the dataset, classifier, and parameters for each candidate.
        candidate_descriptions.extend([{"Dataset": dataset["Dataset"], "Classifier": family, "L2": dataset["L2"], "Digit selection": dataset["Digit selection"], "SVD": dataset["SVD"], "Parameters": json.dumps(combination)} for combination in ParameterGrid(parameters)])


# Summarize the classifier settings included in the comparison.
model_options = pd.DataFrame([{"Classifier": name, "Parameters": ", ".join(key.replace("classifier__", "") + ": " + " / ".join(map(str, values)) for key, values in grid.items()) or "Fixed"} for name, (_, grid) in model_families.items()])

# Display the classifier settings.
display(model_options.style.hide(axis="index"))

# Report the total number of datasets, candidates, and validation fits.
print("Datasets:", len(dataset_options), "| Candidates:", len(candidate_descriptions), "| CV fits:", len(candidate_descriptions) * len(validation_splits))


# Verify that the validation splits match the intended cross-validation design.
assert len(validation_splits) == CV_FOLDS, "Validation splits must match CV_FOLDS."

# Evaluate preprocessing choices and classifiers within the same pipeline.
model_pipeline = Pipeline([("normalize", "passthrough"), ("select", "passthrough"), ("svd", "passthrough"), ("classifier", LogisticRegression())])

# Prioritize macro-F1 to account for uneven occupation-class representation.
scoring = {"macro_f1": "f1_macro", "accuracy": "accuracy"}

# Select the strongest configuration using grouped validation rather than held-out test labels.
search = GridSearchCV(model_pipeline, parameter_grids, cv=validation_splits, scoring=scoring, refit="macro_f1", n_jobs=3, return_train_score=False, error_score="raise", verbose=1)

# Start measuring the full search runtime.
search_start = time.perf_counter()

# Evaluate the candidates and retain warnings for diagnosing training problems.
with warnings.catch_warnings(record=True) as recorded_warnings:
    warnings.simplefilter("always")
    with joblib.parallel_backend("threading"):
        search.fit(development_matrix, development_targets)

# Calculate the total search runtime.
search_seconds = time.perf_counter() - search_start

# Report the measured runtime.
print("Search seconds:", round(search_seconds, 1))

# Preserve the fitted search for later evaluation.
joblib.dump(search, results_directory / "search_checkpoint.joblib")


# Combine candidate descriptions with their cross-validation scores.
model_results = pd.concat([pd.DataFrame(candidate_descriptions), pd.DataFrame(search.cv_results_).drop(columns=["params"] + [name for name in search.cv_results_ if name.startswith("param_")])], axis=1)

# Save every candidate's validation results.
model_results.to_csv(results_directory / "cross_validation_results.csv", index=False, encoding="utf-8-sig")

# Summarize warnings that may indicate unstable or unsuitable configurations.
warning_counts = pd.Series([str(item.message) for item in recorded_warnings], dtype="string").value_counts().rename("Count")

# Save the warning summary for further investigation.
warning_counts.to_csv(results_directory / "training_warnings.csv", encoding="utf-8-sig")


# Identify the configuration with the highest cross-validation macro-F1.
best_configuration = model_results.loc[search.best_index_]

# Retain the strongest parameter setting for each dataset-classifier combination.
family_results = model_results.sort_values("mean_test_macro_f1", ascending=False, kind="stable").drop_duplicates(["Dataset", "Classifier"])

# Compare model performance, variability across folds, and accuracy.
ranked_models = family_results[["Dataset", "Classifier", "L2", "Digit selection", "SVD", "mean_test_macro_f1", "std_test_macro_f1", "mean_test_accuracy"]].rename(columns={"Classifier": "Model", "mean_test_macro_f1": "Macro F1", "std_test_macro_f1": "Fold SD", "mean_test_accuracy": "Accuracy"})

# Number the combinations according to their validation ranking.
ranked_models.insert(0, "Rank", range(1, len(ranked_models) + 1))

# Display the complete ranking with consistent numerical precision.
display(ranked_models.style.hide(axis="index").format({"Macro F1": "{:.3f}", "Fold SD": "{:.3f}", "Accuracy": "{:.1%}"}))

# Report the selected feature representation.
print("Selected representation:", "L2 =", best_configuration["L2"], "| Digit selection =", best_configuration["Digit selection"], "| SVD =", best_configuration["SVD"])

# Report the winning classifier and its validation performance.
print("Selected model:", best_configuration["Classifier"], "| CV macro F1:", f'{best_configuration["mean_test_macro_f1"]:.3f}', "| CV accuracy:", f'{best_configuration["mean_test_accuracy"]:.1%}')

### Original experiment — historical results (preserved)

The earlier notebook recorded **229 configurations × 10 grouped folds = 2,290 cross-validation fits**, with **1,163.7 seconds** of search time. Its reported selected configuration was **Random Forest + L2 normalization + cluster memberships**, with development macro-F1 **0.481**, held-out accuracy **0.579 (22/38)**, and held-out macro-F1 **0.471**. These are historical outputs from the original experiment, **not results from the new three-fold comparison**. Preserve the original notebook and outputs separately for reproducibility. The current experiment does not recreate its fold-local feature transformations or cluster features, so the scores are not directly comparable.


### Compare datasets before interpreting classifiers

The first table lists every dataset representation and its strongest classifier. The second lists every classifier within each dataset. This separates the effect of representation choices from the choice of model. The plot shows the leading dataset–classifier combinations; fold-level results remain visible. All representation learning takes place inside training folds, but the original questionnaire feature preparation still predates splitting.


In [ ]:
# Prepare interactive Plotly charts and 4K image export in Google Colab.
import sys
import subprocess
import importlib.util

# Install missing visualization packages.
if importlib.util.find_spec("plotly") is None or importlib.util.find_spec("kaleido") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "plotly", "kaleido"])

# Load Plotly and configure the Colab renderer.
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "colab"

# Ensure the output directories exist.
plot_directory.mkdir(parents=True, exist_ok=True)
results_directory.mkdir(parents=True, exist_ok=True)


# Save every chart in interactive HTML and 4K PNG formats.
def save_chart(figure, name):
    figure.show()
    figure.write_html(plot_directory / f"{name}.html", include_plotlyjs=True)
    figure.write_image(plot_directory / f"{name}_4k.png", width=3840, height=2160)


# Compare the strongest classifier for each dataset representation.
dataset_results = ranked_models.sort_values("Macro F1", ascending=False).drop_duplicates("Dataset")

# Display the winning classifier and validation metrics for each dataset.
display(dataset_results[["Dataset", "Model", "L2", "Digit selection", "SVD", "Macro F1", "Fold SD", "Accuracy"]].style.hide(axis="index").format({"Macro F1": "{:.3f}", "Fold SD": "{:.3f}", "Accuracy": "{:.1%}"}))

# Save the dataset comparison.
dataset_results.to_csv(results_directory / "dataset_comparison.csv", index=False, encoding="utf-8-sig")


# Compare the 18 strongest dataset-classifier combinations.
plot_results = ranked_models.nlargest(18, "Macro F1").copy()

# Identify each combination by its dataset and classifier.
plot_results["Variant"] = plot_results["Dataset"] + " | " + plot_results["Model"]

# Place the strongest combination at the top.
plot_results = plot_results.sort_values("Macro F1")

# Show macro-F1 alongside accuracy and fold variability.
figure = px.bar(plot_results, x="Macro F1", y="Variant", orientation="h", color="Macro F1", color_continuous_scale="Teal", text="Macro F1", hover_data={"Dataset": True, "Model": True, "Fold SD": ":.3f", "Accuracy": ":.1%", "Macro F1": ":.3f", "Variant": False})

# Preserve the complete macro-F1 scale.
figure.update_xaxes(range=[0, 1.05], title="Mean cross-validation macro-F1", tickformat=".1f")

# Keep dataset and classifier labels readable.
figure.update_yaxes(title=None, automargin=True)

# Display exact validation scores.
figure.update_traces(texttemplate="%{x:.3f}", textposition="outside", cliponaxis=False)

# Apply a consistent presentation layout.
figure.update_layout(title="Occupation classifier comparison", template="plotly_white", height=850, margin=dict(l=50, r=100, t=90, b=60), coloraxis_showscale=False, font=dict(size=15))

# Display and export the comparison in interactive and 4K formats.
save_chart(figure, "classifier_comparison")


# Investigate the selected classifier's performance across validation folds.
fold_overview = pd.DataFrame([{"Fold": fold, "Training records": len(fit_rows), "Validation records": len(validation_rows), "Unseen-class records": int((~development_targets.iloc[validation_rows].isin(development_targets.iloc[fit_rows])).sum()), "Macro F1": search.cv_results_[f"split{fold - 1}_test_macro_f1"][search.best_index_], "Accuracy": search.cv_results_[f"split{fold - 1}_test_accuracy"][search.best_index_]} for fold, (fit_rows, validation_rows) in enumerate(validation_splits, start=1)])

# Preserve the fold-level results.
fold_overview.to_csv(results_directory / "selected_fold_results.csv", index=False, encoding="utf-8-sig")

# Display fold performance and unseen-class counts.
display(fold_overview.style.hide(axis="index").format({"Macro F1": "{:.3f}", "Accuracy": "{:.1%}"}))

# Prepare readable fold labels.
fold_overview["Fold label"] = "Fold " + fold_overview["Fold"].astype(str)

# Compare validation scores and inspect class coverage on hover.
figure = px.bar(fold_overview, x="Fold label", y="Macro F1", color="Macro F1", color_continuous_scale="Teal", text="Macro F1", hover_data={"Training records": True, "Validation records": True, "Unseen-class records": True, "Accuracy": ":.1%", "Macro F1": ":.3f"})

# Use the same macro-F1 scale as the classifier comparison.
figure.update_yaxes(range=[0, 1.05], title="Validation macro-F1", tickformat=".1f")

# Display exact scores above each fold.
figure.update_traces(texttemplate="%{y:.3f}", textposition="outside", cliponaxis=False)

# Keep the fold comparison readable.
figure.update_layout(title="Selected classifier: validation fold performance", template="plotly_white", height=600, margin=dict(l=60, r=60, t=90, b=60), coloraxis_showscale=False, xaxis_title=None, font=dict(size=15))

# Display and export the fold comparison.
save_chart(figure, "selected_fold_comparison")


# Investigate features associated with a single occupation-code digit.
digit_diagnostic = DigitExclusiveSelector(min_support=3).fit(development_matrix, development_targets)

# Record the observed support and digit exclusivity of each feature.
exclusive_features = pd.DataFrame({"Feature": feature_names, "Support": digit_diagnostic.support_, "Digit exclusive": digit_diagnostic.exclusive_})

# Retain features satisfying the digit-exclusivity rule.
exclusive_features = exclusive_features.loc[exclusive_features["Digit exclusive"]].sort_values("Support", ascending=False)

# Display the 25 most frequently observed exclusive features.
display(exclusive_features.head(25).style.hide(axis="index"))

# Select the 20 features with the highest observed support.
feature_plot = exclusive_features.head(20).sort_values("Support").copy()

# Visualize feature support when qualifying features exist.
if not feature_plot.empty:

    # Compare feature occurrence counts without treating them as predictive importance.
    figure = px.bar(feature_plot, x="Support", y="Feature", orientation="h", color="Support", color_continuous_scale="Teal", text="Support", hover_data={"Feature": True, "Support": True})

    # Explain that support counts development records.
    figure.update_xaxes(title="Development records containing the feature", rangemode="tozero", dtick=1)

    # Preserve readable feature names.
    figure.update_yaxes(title=None, automargin=True)

    # Display exact support counts.
    figure.update_traces(textposition="outside", cliponaxis=False)

    # Apply the same visual style as the other charts.
    figure.update_layout(title="Features associated with a single occupation-code digit", template="plotly_white", height=850, margin=dict(l=50, r=100, t=90, b=60), coloraxis_showscale=False, font=dict(size=15))

    # Display and export the feature-support chart.
    save_chart(figure, "digit_exclusive_features")

else:

    # Report when no features meet the minimum support requirement.
    print("No digit-exclusive features met the minimum support requirement.")

### Evaluate the selected model once on the reserved test

The three grouped development folds select the representation, classifier, and hyperparameters. Only that selected pipeline is evaluated on held-out records. This avoids using test performance to choose among candidates. Compare its predictions with the majority baseline, inspect errors and occupation-code prefixes, and keep unseen classes in the evaluation denominator.


In [ ]:
# Supply a score ranking only when the fitted classifier exposes one.
def prediction_scores(fitted_pipeline, inputs):
    if hasattr(fitted_pipeline, "predict_proba"):
        return fitted_pipeline.predict_proba(inputs), "Model probabilities (uncalibrated)"
    if hasattr(fitted_pipeline, "decision_function"):
        scores = np.asarray(fitted_pipeline.decision_function(inputs))
        return scores if scores.ndim == 2 else np.column_stack([-scores, scores]), "Decision scores (not probabilities)"
    return None, "Unavailable"


### Selected classifier on the reserved test

The overall configuration was selected using development cross-validation. Report its accuracy, macro F1, and agreement at one through four occupation-code digits. Classes absent from development remain in the denominator. These measurements compare predictions with the supplied manual labels, which are not independently adjudicated ground truth.

In [ ]:
# Evaluate the selected classifier on held-out records excluded from model selection.
selected_pipeline = search.best_estimator_

# Predict complete four-digit occupation codes.
test_predictions = np.asarray(selected_pipeline.predict(test_matrix)).astype(str).ravel()

# Standardize recorded labels for comparisons.
actual_targets = np.asarray(test_targets).astype(str).ravel()

# Compare against the most frequent occupation in development data.
baseline = DummyClassifier(strategy="most_frequent").fit(development_matrix, development_targets)

# Predict the held-out records using the baseline.
baseline_predictions = np.asarray(baseline.predict(test_matrix)).astype(str).ravel()


# Retain original questionnaire responses for reviewing prediction errors.
held_out_predictions = raw_records.loc[test_predictors.index, ["ID", "ShemAvoda", "EzoAvoda"]].copy()

# Attach actual and predicted occupation codes.
held_out_predictions["Recorded target"] = actual_targets
held_out_predictions["Prediction"] = test_predictions

# Distinguish previously observed occupations from unseen classes.
held_out_predictions["Class seen in development"] = np.isin(actual_targets, np.asarray(development_targets).astype(str))

# Identify correct predictions.
held_out_predictions["Correct"] = actual_targets == test_predictions

# Separate errors on unseen classes from errors on known classes.
held_out_predictions["Error type"] = np.select([held_out_predictions["Correct"], ~held_out_predictions["Class seen in development"]], ["Correct", "Unseen occupation"], default="Incorrect known occupation")


# Compare held-out accuracy and macro-F1 with the majority baseline.
held_out_metrics = pd.DataFrame([{"Model": best_configuration["Classifier"], "Accuracy": accuracy_score(actual_targets, test_predictions), "Macro F1": f1_score(actual_targets, test_predictions, average="macro", zero_division=0)}, {"Model": "Majority baseline", "Accuracy": accuracy_score(actual_targets, baseline_predictions), "Macro F1": f1_score(actual_targets, baseline_predictions, average="macro", zero_division=0)}])

# Save and display the held-out model comparison.
held_out_metrics.to_csv(results_directory / "held_out_metrics.csv", index=False, encoding="utf-8-sig")
display(held_out_metrics.style.hide(axis="index").format({"Accuracy": "{:.1%}", "Macro F1": "{:.3f}"}))


# Measure agreement across successive occupation-code prefixes.
prefix_metrics = pd.DataFrame([{"Digits": width, "Agreement": np.mean([predicted[:width] == actual[:width] for predicted, actual in zip(test_predictions, actual_targets)])} for width in range(1, 5)])

# Save and display hierarchical agreement.
prefix_metrics.to_csv(results_directory / "hierarchical_agreement.csv", index=False, encoding="utf-8-sig")
display(prefix_metrics.style.hide(axis="index").format({"Agreement": "{:.1%}"}))


# Measure agreement at each individual occupation-code position.
digit_metrics = pd.DataFrame([{"Digit": position + 1, "Accuracy": np.mean([predicted[position] == actual[position] for predicted, actual in zip(test_predictions, actual_targets)])} for position in range(4)])

# Save and display digit-level accuracy.
digit_metrics.to_csv(results_directory / "digit_accuracy.csv", index=False, encoding="utf-8-sig")
display(digit_metrics.style.hide(axis="index").format({"Accuracy": "{:.1%}"}))


# Inspect class-level precision, recall, F1, and support.
classification_details = pd.DataFrame(classification_report(actual_targets, test_predictions, output_dict=True, zero_division=0)).T

# Save and display the classification report.
classification_details.to_csv(results_directory / "classification_report.csv", encoding="utf-8-sig")
display(classification_details.round(3))


# Report how many held-out records belong to occupations absent from development data.
unseen_count = int((~held_out_predictions["Class seen in development"]).sum())
print("Held-out records:", len(actual_targets))
print("Records with unseen occupations:", unseen_count)
print("Distinct unseen occupations:", len(set(actual_targets) - set(np.asarray(development_targets).astype(str))))


# Retrieve class scores when the selected model supports ranking alternatives.
test_scores, test_score_kind = prediction_scores(selected_pipeline, test_matrix)

# Evaluate whether the correct occupation appears among the top three alternatives.
if test_scores is not None:

    # Match score columns to their occupation labels.
    score_classes = np.asarray(selected_pipeline.classes_).astype(str)

    # Rank up to three candidate occupations for each record.
    top_three = score_classes[np.argsort(-test_scores, axis=1, kind="stable")[:, :min(3, len(score_classes))]]

    # Record the alternatives and whether they contain the correct occupation.
    held_out_predictions["Top three"] = [", ".join(values) for values in top_three]
    held_out_predictions["Top-three correct"] = [actual in alternatives for actual, alternatives in zip(actual_targets, top_three)]

    # Report top-three accuracy without assuming the scores are calibrated probabilities.
    print("Top-three accuracy:", f'{held_out_predictions["Top-three correct"].mean():.1%}')

# Identify the type of score used for ranking.
held_out_predictions["Score kind"] = test_score_kind

# Save and display held-out predictions for manual review.
held_out_predictions.to_csv(results_directory / "held_out_predictions.csv", index=False, encoding="utf-8-sig")
display(held_out_predictions)


# Examine incorrect classifications alongside the original occupation descriptions.
incorrect_records = raw_records.loc[held_out_predictions.index[~held_out_predictions["Correct"]], ["ID", "EzoAvoda", "TeurPeula", "TeurTafkid", "SemelMishlachSofi"]].join(held_out_predictions[["Prediction", "Class seen in development", "Error type"]])

# Display records requiring error analysis.
display(incorrect_records)


### Interpretation of the classifier comparison

All candidates use the same prepared features. The selected model is evaluated once on the held-out description groups. This does not eliminate the earlier preprocessing leakage; the scores remain exploratory until learned field transformations are fitted using training data only.


*No additional feature processing or second validation experiment is performed here.*


*No additional feature processing or second validation experiment is performed here.*


**Evaluation scope:** The three-fold model search and the reserved-group test operate on the completed feature matrix. Results from older random-row, SVD, feature-selection, and clustering experiments are not part of this run. The scores remain exploratory because feature preparation was completed before splitting.


### Final fit and predictions

Refit the single selected configuration on all complete labels and predict the nine incomplete records. Retain each original partial code. A compatible alternative uses only classes learned during training and is reported separately from the unconstrained prediction. Agreement with known digits does not establish correctness of the unknown digits. These predictions support examination by a coder, and the simulated sample does not establish a safe automation threshold.

In [ ]:
# How can incomplete occupation codes be predicted without contradicting known digits? Refit on complete labels and check each partial-code constraint.
final_pipeline = clone(selected_pipeline).fit(training_matrix, labelled_targets)
unresolved_predictors = prediction_matrix
final_predictions = np.asarray(final_pipeline.predict(prediction_matrix)).ravel()
prediction_results = raw_records.loc[~complete_target, ["ID", "ShemAvoda", "EzoAvoda"]].copy()
prediction_results["Recorded target"] = occupation_codes.loc[~complete_target]
prediction_results["Known prefix"] = known_prefixes
prediction_results["Predicted occupation"] = final_predictions
prediction_results["Prefix agrees"] = [prediction.startswith(prefix) for prediction, prefix in zip(final_predictions, known_prefixes)]


partial_scores, partial_score_kind = prediction_scores(final_pipeline, prediction_matrix)
trained_codes = final_pipeline.classes_
ranking_rows = []
for row_number, (index, record) in enumerate(prediction_results.iterrows()):
    compatible = np.array([all(original == "X" or original == proposed for original, proposed in zip(record["Recorded target"], code)) for code in trained_codes])
    ranking = np.argsort(-partial_scores[row_number], kind="stable") if partial_scores is not None else np.array([], dtype=int)
    compatible_ranking = ranking[compatible[ranking]]
    ranking_rows.append({"ID": int(record["ID"]), "Top three": ", ".join(trained_codes[ranking[:3]]) if len(ranking) else "", "Score kind": partial_score_kind, "Maximum score": float(partial_scores[row_number].max()) if partial_scores is not None else np.nan, "Digit-compatible alternative": trained_codes[compatible_ranking[0]] if len(compatible_ranking) else pd.NA, "Compatible learned classes": int(compatible.sum())})
prediction_results = prediction_results.merge(pd.DataFrame(ranking_rows), on="ID", how="left")
prediction_results.to_csv(results_directory / "incomplete_label_predictions.csv", index=False, encoding="utf-8-sig")
display(prediction_results)


### Predict the nine incomplete occupation codes

Use `prediction_matrix`, built during the same one-pass column preparation. The final classifier is refitted on all 191 complete-label feature rows. Its predictions are checked against known digits, without repeating any field transformations.


In [ ]:
# The final classifier expects the already transformed columns, not a raw questionnaire.
print("Final classifier input features:", training_matrix.shape[1])
print("Incomplete records awaiting classification:", prediction_matrix.shape[0])
display(prediction_results[["ID", "Recorded target", "Predicted occupation", "Digit-compatible alternative"]])


### What the saved model contains

The classifier expects the already transformed numerical columns. The fitted encoders and vectorizers saved during the earlier field sections are needed to prepare any future raw questionnaire. No preprocessing is repeated in the modeling section.


In [ ]:
# What must be saved so later predictions use the same feature definitions? Persist the fitted model and preprocessing artifacts.
joblib.dump(final_pipeline, results_directory / "selected_pipeline.joblib")
pd.Series(feature_names, name="Feature").to_csv(results_directory / "selected_feature_names.csv", index=False, encoding="utf-8-sig")
run_manifest = {"seed": SEED, "training_records": len(labelled_predictors), "development_records": len(development_predictors), "test_records": len(test_predictors), "incomplete_records": len(prediction_results), "folds": len(validation_splits), "candidates": len(model_results), "selected_classifier": best_configuration["Classifier"], "selected_normalization": "l2" if best_configuration["L2"] else None, "selected_digit_selection": bool(best_configuration["Digit selection"]), "selected_svd": bool(best_configuration["SVD"]), "selected_parameters": json.loads(best_configuration["Parameters"]), "validation_macro_f1": float(best_configuration["mean_test_macro_f1"]), "test_accuracy": float(held_out_metrics.iloc[0]["Accuracy"]), "test_macro_f1": float(held_out_metrics.iloc[0]["Macro F1"]), "final_features": training_matrix.shape[1], "tenure_reference_date": tenure_reference_date, "management_missing_value": 0, "employment_missing_category": 0, "salary_missing_category": 4, "qualification_missing_category": 9, "text_transformations": text_transformations, "industry_in_primary_model": False, "sklearn_version": sklearn.__version__, "search_seconds": search_seconds}
(results_directory / "run_manifest.json").write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Selected pipeline, validation results, and final predictions saved.")


## Results and conclusion

The configuration selected by development cross-validation determines the final model. Compare its held-out agreement with the majority baseline and inspect performance at each occupation-code prefix. The dataset-first comparison evaluates all eight representation specifications using the same grouped folds, then compares compatible classifiers within each dataset. Final predictions for incomplete labels are distinct from measured test performance because their complete reference labels are unavailable.


In [ ]:
# Which errors occur at the occupation-prefix level and which representations contribute? Inspect these diagnostics to guide improvement.
figure, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout="constrained")
bars = axes[0].bar(prefix_metrics["Digits"].astype(str), prefix_metrics["Agreement"], color="#234737")
axes[0].bar_label(bars, labels=[f"{value:.1%}" for value in prefix_metrics["Agreement"]], padding=4)
axes[0].set(xlabel="Occupation prefix length", ylabel="Agreement", ylim=(0, 1.08), title="Selected model: held-out agreement")
positions = np.arange(len(held_out_metrics))
for offset, metric, color in [(-0.18, "Accuracy", "#234737"), (0.18, "Macro F1", "#946E37")]:
    bars = axes[1].bar(positions + offset, held_out_metrics[metric], width=0.34, color=color, label=metric)
    axes[1].bar_label(bars, fmt="%.3f", padding=4)
axes[1].set(xticks=positions, xticklabels=held_out_metrics["Model"], ylim=(0, 1.08), title="Selected model and majority baseline")
axes[1].legend(frameon=False)
figure.savefig(plot_directory / "held_out_summary.png", dpi=180, facecolor="#F8F7F1")
display(Image(filename=str(plot_directory / "held_out_summary.png")))
plt.close(figure)


**Conclusion:** Eight dataset specifications are enumerated before classifier combinations, and every compatible dataset–classifier pairing is evaluated with three grouped folds. No questionnaire columns are reprocessed. Target-digit selection and SVD are fitted inside the training portion of each fold. The earlier 10-fold experiment remains preserved separately. Because initial questionnaire feature preparation preceded splitting, the current scores are still exploratory.
